# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from tqdm.auto import tqdm
from os.path import join

# Import Functions
sys.path.append("../../")
from src.configs.default_configs import fn_pred, fn_pred_perf
from src.configs.blood_config import data_name, batch_size, eval_batch_size, \
    data_name_ood_in, data_name_ood_out, data_name_ood_diff, seed_interval_size, ensemble_size

from src.file_manager.filepath import FilePath

from src.training.misc import get_class_weights
from src.evaluation.inference import split_test_set
from src.file_manager.load_save_df import save_pred_df, load_pred_df, save_pred_perf_df

from src.evaluation.evaluate import get_model_performance
from src.data_generator.blood import load_bloodmnist_data_dict
from src.data_generator.raabin import load_raabin_data_dict
from src.data_generator.bonemarrow import load_bonemarrow_data_dict
from src.data_generator.chest_mnist import load_chestmnist_data_dict
from src.data_processing.ood_dataset_preprocessing import process_dataset_for_ood, left_join_datasets
from src.models.resnet.model import ResNet18
from src.models.resnet.train import train_resnet
from src.models.de.train import train_ensemble_w_best_param
from src.models.de.predict import get_all_ensemble_predictions, get_all_ensemble_predictions_time
from src.models.de.model import DeepEnsemble
from src.data_processing.dataloader import get_pytorch_split_dict_image
from src.models.resnet.predict import get_resnet_predictions_speedup, get_resnet_predictions

from cur_seed import seed
# seed = 2024
cur_model_name="tuned"

fp = FilePath(data_name=data_name, seed=seed)
fp_ood_in = FilePath(data_name=data_name_ood_in, seed=seed)
fp_ood_out = FilePath(data_name=data_name_ood_out, seed=seed)
fp_ood_diff = FilePath(data_name=data_name_ood_diff, seed=seed)

# Get Data

In [ ]:
print("Loading Data Dict")
data_dict = load_bloodmnist_data_dict(fp_preprocessed=fp.get_preprocessed_folder())
num_ori_test = len(data_dict["test_df"])

# Add ID Class OOD into the Test Set
print("Loading ID Class OOD Data Dict")
data_dict_ood_in = load_raabin_data_dict(fp_preprocessed=fp_ood_in.get_preprocessed_folder())
data_dict_ood_in = process_dataset_for_ood(data_dict, data_dict_ood_in, seed)

# ODD Class OOD
print("Loading OOD Class OOD Data Dict")
data_dict_ood_out = load_bonemarrow_data_dict(fp_preprocessed=fp_ood_out.get_preprocessed_folder())
data_dict_ood_out = process_dataset_for_ood(data_dict, data_dict_ood_out, seed)

# OOD Modality
print("Loading OOD Modality OOD Data Dict")
data_dict_ood_diff = load_chestmnist_data_dict(
    fp_preprocessed=fp_ood_diff.get_preprocessed_folder(), only_test=True, num_classes=8)
data_dict_ood_diff = process_dataset_for_ood(data_dict, data_dict_ood_diff, seed)

data_dict = left_join_datasets(data_dict, data_dict_ood_in)

# Training

In [ ]:
class_weights = get_class_weights(data_dict)
params = dict(
    ModelClass=ResNet18,
    data_dict=data_dict,
    batch_size=batch_size,
    eval_batch_size=eval_batch_size,
    train_model_func=train_resnet,
    metric_to_monitor="ce loss",
    maximise=False,
    seed=seed,
    train_param_dict = dict(
        max_epochs=500, weight_decay=0.001, patience=5, lr=0.001, class_weights=class_weights),
    pytorch_split_dict_func=get_pytorch_split_dict_image
)
train_ensemble_w_best_param(
    **params,
    best_param={},
    cur_model_name=cur_model_name,
    seed_interval_size=seed_interval_size, ensemble_size=ensemble_size,
    data_name=data_name
)

# Prediction

In [ ]:
pred_df = get_all_ensemble_predictions(
    data_dict, pred_func=get_resnet_predictions_speedup, 
    batch_size=batch_size, eval_batch_size=eval_batch_size, 
    data_name=data_name, seed=seed, 
    ModelClass=ResNet18, cur_model_name=cur_model_name,
    seed_interval_size=seed_interval_size, ensemble_size=ensemble_size,
    additional_pred_args={"mc":False},
    model_label="resnet",
)
pred_df = split_test_set(
    pred_df, split_col="split", new_split_col="split_perf", 
    num_ori_test=num_ori_test, labels=["Test-Blood", "Test-Raabin"])
save_pred_df(pred_df=pred_df, fp=fp, ModelClass=DeepEnsemble)

# Performance Evaluation

In [ ]:
pred_df = load_pred_df(fp=fp, ModelClass=DeepEnsemble)
perf_df = get_model_performance(
    all_pred_df=pred_df, data_dict=data_dict, label="de", perf_split_col="split_perf")
save_pred_perf_df(pred_perf_df=perf_df, fp=fp, ModelClass=DeepEnsemble)
perf_df

# OOD Prediction

In [ ]:
ood_dicts = {
    "ood_in_raabin": data_dict_ood_in, 
    "ood_out_bonemarrow": data_dict_ood_out,
    "ood_chestmnist": data_dict_ood_diff}
for label, cur_ood_data_dict in tqdm(ood_dicts.items(), total=len(ood_dicts)):
    pred_df_ood = get_all_ensemble_predictions(
        cur_ood_data_dict, pred_func=get_resnet_predictions_speedup, 
        batch_size=batch_size, eval_batch_size=eval_batch_size, 
        data_name=data_name, seed=seed, 
        ModelClass=ResNet18, cur_model_name="tuned",
        seed_interval_size=100, ensemble_size=5,
        additional_pred_args={"mc":False},
        model_label="resnet",
        optional_label=label, #override=True
    )
    save_pred_df(pred_df=pred_df_ood, fp=fp, ModelClass=DeepEnsemble, optional_label=label)